# DocAgent on Kaggle: data, B1/B2/B3 baselines, results

**Settings (right panel):** Accelerator **GPU T4 x2**, Internet **On**. Attach your `manuals` dataset (Add Input → Your Datasets).

**How to run: one click, then close the tab.**
Click **Save Version → Save & Run All (Commit)**. Kaggle runs every cell in the background (about 3 hours the first time, under 2 hours once data is saved). You don't need to keep the browser open or run cells one by one.

**When it finishes:** open the version → **Output**. `summary.md` has the results table, `errors.txt` lists failures to read.

**Next time:** Add Input → Your Work → this notebook. The setup cell restores the data, so only missing parts are redone.

Working interactively instead? Always run the Setup cell first: every later cell uses its helpers.

## Setup

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

os.environ["PYTHONUNBUFFERED"] = "1"   # show script output live, not only at the end
WORK = Path("/kaggle/working")
REPO = WORK / "DocAgent"
os.chdir(WORK)

def _copy_missing(src, dst):
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
    return dst

# Repair: an old "git clone + %cd" cell may have nested copies (DocAgent/DocAgent/...).
# Keep the innermost repo, merge any data found in the outer copies, and move it to the top.
inner = REPO
while (inner / "DocAgent" / "docagent").is_dir():
    inner = inner / "DocAgent"
if inner != REPO:
    print(f"Repairing nested folders: keeping {inner}")
    level = inner.parent
    while level != WORK:
        for part in ("data", "results"):
            if (level / part).is_dir():
                shutil.copytree(level / part, inner / part, dirs_exist_ok=True, copy_function=_copy_missing)
        level = level.parent
    shutil.move(str(inner), str(WORK / "_DocAgent_fixed"))
    shutil.rmtree(REPO)
    shutil.move(str(WORK / "_DocAgent_fixed"), str(REPO))

if not REPO.is_dir():
    subprocess.run(["git", "clone", "-q", "https://github.com/ademghorbel/DocAgent.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "pull", "-q"], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Working in", os.getcwd())

def sh(cmd):
    """Run a shell command with live output; stop the cell if it fails."""
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"Command failed (exit {code}): {cmd}")

def have(path):
    return Path(path).exists()

sh("pip install -q -r requirements.txt")
sh("python scripts/restore_data.py")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

## Settings

In [ ]:
SYSTEMS = "B1 B2 B3"   # B3 = ColQwen page retrieval -> VLM (week 2)
RUN_NAME = "baselines"   # results/<RUN_NAME>/; a rerun with the same name skips answered questions
LIMIT = 0                # >0: only the first N questions per file (quick smoke test, use another RUN_NAME)


## Status

In [ ]:
import json
def status():
    for name in ("mmlongbench", "vidore_industrial", "manuals"):
        qf = Path(f"data/questions/{name}.jsonl") if name != "manuals" else Path("data/questions/own.jsonl")
        pf = Path(f"data/pages/{name}/pages.jsonl")
        nq = sum(1 for _ in open(qf)) if qf.exists() else 0
        npg = sum(1 for _ in open(pf)) if pf.exists() else 0
        print(f"{name:18s} questions: {nq:4d}   pages: {npg:5d}")
status()

## Tests (CPU, a few seconds)

In [ ]:
sh("python -m pytest -q")

## Data (each step skips itself when already done)
MMLongBench ~5 min, ViDoRe 15–20 min (5,244 pages), manuals ~15 min, OCR ~5 min. ViDoRe only counts as done when all its pages are saved: a cut-off run resumes.

In [ ]:
if have("data/questions/mmlongbench.jsonl") and have("data/pages/mmlongbench/pages.jsonl"):
    print("MMLongBench ready.")
else:
    sh("python scripts/prepare_mmlongbench.py --n 100 --max-docs 15")

if have("data/pages/vidore_industrial/COMPLETE"):
    print("ViDoRe ready.")
else:
    sh("python scripts/prepare_vidore.py --n 60")

if have("data/pages/manuals/pages.jsonl"):
    print("Manuals ready.")
else:
    sh("python scripts/download_manuals.py --from /kaggle/input")

sh("apt-get install -y -qq tesseract-ocr > /dev/null && pip install -q pytesseract")
sh("python scripts/ocr_scanned.py")
status()


## Own questions

In [ ]:
if have("data/questions/own.jsonl"):
    sh("python scripts/check_questions.py data/questions/own.jsonl")
else:
    print("No own.jsonl yet: the run uses the two benchmarks only.")
QUESTIONS = " ".join(f for f in ["data/questions/mmlongbench.jsonl", "data/questions/vidore_industrial.jsonl",
                                 "data/questions/own.jsonl"] if have(f))
print("Question files:", QUESTIONS)


## ColQwen page index for B3 (~25 min the first time, resumes if cut off)

In [ ]:
if "B3" in SYSTEMS:
    try:
        sh(f"python scripts/index_pages.py --questions {QUESTIONS}")
    except RuntimeError as e:   # don't lose the B1/B2 run over a B3 problem
        print("ColQwen index failed, running without B3:", e)
        SYSTEMS = SYSTEMS.replace("B3", "").strip()


## Full run: B1, B2, B3 with the LLM judge (finished questions are skipped on a rerun)

In [ ]:
use_wandb = ""
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    use_wandb = "--wandb"
except Exception:
    pass
limit = f"--limit {LIMIT}" if LIMIT else ""
sh(f"python eval/run_eval.py --questions {QUESTIONS} --systems {SYSTEMS} --judge llm --run {RUN_NAME} {limit} {use_wandb}")


## Results and errors (also copied to the top of Output)

In [ ]:
from IPython.display import Markdown, display
summary = Path(f"results/{RUN_NAME}/summary.md")
display(Markdown(summary.read_text()))
shutil.copy(summary, WORK / "summary.md")
sh(f"python eval/show_errors.py --run {RUN_NAME} --questions {QUESTIONS} -n 15 > {WORK}/errors.txt")
print(open(WORK / "errors.txt").read())
